# 🔵 RAG on Intel GPUs with Haystack and vLLM

<img src="https://haystack.deepset.ai/images/haystack-ogimage.png" width="400" style="display:inline;">

This notebook builds a complete **Retrieval-Augmented Generation (RAG)** pipeline where **every model runs on Intel® GPUs** (Intel® Arc™ or Intel® Data Center GPU Max series), served locally with [vLLM](https://docs.vllm.ai/).

We self-host three models, each on its own Intel GPU, all exposing OpenAI-compatible endpoints:

| Role | Model | Port | Haystack component |
|------|-------|------|--------------------|
| Embeddings | `sentence-transformers/all-MiniLM-L6-v2` | 8001 | `VLLMDocumentEmbedder` / `VLLMTextEmbedder` |
| Ranking | `BAAI/bge-reranker-base` | 8002 | `VLLMRanker` |
| Generation | `Qwen/Qwen2.5-1.5B-Instruct` | 8000 | `VLLMChatGenerator` |

Because vLLM exposes the same API on every backend, the Haystack pipeline code is **hardware-agnostic** — the exact same notebook runs on NVIDIA GPUs by launching the servers differently.

## 1. Serve the models on Intel GPUs

We serve three models with vLLM on Intel GPUs. **Full setup instructions, requirements, and Intel-specific gotchas are in [`scripts/README_intel_gpu.md`](../scripts/README_intel_gpu.md)**, and the [`scripts/setup_vllm_xpu.sh`](../scripts/setup_vllm_xpu.sh) helper launches each server.

Run the commands below on a machine with Intel GPUs (not inside this notebook if it's on Colab). Each server is pinned to a different GPU via `ZE_AFFINITY_MASK`.

In [1]:
# Serve the three models on Intel GPUs using the helper script.
# See scripts/README_intel_gpu.md for details and gotchas (proxy, ZE_AFFINITY_MASK, --task embed).
#
#   # Generation model on GPU 0, port 8000
#   ./scripts/setup_vllm_xpu.sh
#
#   # Embedding model on GPU 1, port 8001
#   MODEL=sentence-transformers/all-MiniLM-L6-v2 TASK=embed \
#     PORT=8001 ZE_AFFINITY_MASK=1 GPU_MEM=0.3 ./scripts/setup_vllm_xpu.sh
#
#   # Ranking model on GPU 2, port 8002
#   MODEL=BAAI/bge-reranker-base \
#     PORT=8002 ZE_AFFINITY_MASK=2 GPU_MEM=0.3 ./scripts/setup_vllm_xpu.sh

EMBED_URL = "http://localhost:8001/v1"
RANK_URL  = "http://localhost:8002/v1"
CHAT_URL  = "http://localhost:8000/v1"
print("Assuming three vLLM servers are running on Intel GPUs (see scripts/README_intel_gpu.md).")

Assuming three vLLM servers are running on Intel GPUs (see scripts/README_intel_gpu.md).


In [ ]:
# Install the Haystack + vLLM integration
! pip install haystack-ai vllm-haystack

## 2. Index documents (embeddings on Intel GPU)

We embed a small knowledge base with `VLLMDocumentEmbedder`, which calls the embedding server on GPU 1, and store the vectors in an in-memory document store.

In [3]:
from haystack import Document, Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack_integrations.components.embedders.vllm import VLLMDocumentEmbedder

documents = [
    Document(content="The Eiffel Tower is located in Paris and was completed in 1889."),
    Document(content="The Great Wall of China is over 13,000 miles long."),
    Document(content="Mount Everest is the tallest mountain, at 8,849 meters."),
    Document(content="The Colosseum in Rome could hold up to 80,000 spectators."),
    Document(content="Paris is the capital of France and home to the Louvre museum."),
]

document_store = InMemoryDocumentStore(embedding_similarity_function="cosine")

document_embedder = VLLMDocumentEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2",
    api_base_url=EMBED_URL,
)
docs_with_embeddings = document_embedder.run(documents)["documents"]
document_store.write_documents(docs_with_embeddings)
print(f"Indexed {document_store.count_documents()} documents (embeddings computed on Intel GPU).")

Calculating embeddings: 0it [00:00, ?it/s]

Calculating embeddings: 1it [00:00,  2.14it/s]

Calculating embeddings: 1it [00:00,  2.12it/s]

Indexed 5 documents (embeddings computed on Intel GPU).


## 3. Build the RAG pipeline

The query flows through four Intel-GPU-backed stages plus retrieval:

`VLLMTextEmbedder` → `InMemoryEmbeddingRetriever` → `VLLMRanker` → `ChatPromptBuilder` → `VLLMChatGenerator`

In [4]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage
from haystack_integrations.components.embedders.vllm import VLLMTextEmbedder
from haystack_integrations.components.rankers.vllm import VLLMRanker
from haystack_integrations.components.generators.vllm import VLLMChatGenerator

template = [ChatMessage.from_user(
    "Answer the question using ONLY the context below.\n\n"
    "Context:\n{% for d in documents %}- {{ d.content }}\n{% endfor %}\n"
    "Question: {{ query }}\nAnswer:"
)]

rag = Pipeline()
rag.add_component("text_embedder", VLLMTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2", api_base_url=EMBED_URL))
rag.add_component("retriever", InMemoryEmbeddingRetriever(document_store=document_store, top_k=4))
rag.add_component("ranker", VLLMRanker(model="BAAI/bge-reranker-base", api_base_url=RANK_URL, top_k=2))
rag.add_component("prompt", ChatPromptBuilder(template=template, required_variables=["query", "documents"]))
rag.add_component("llm", VLLMChatGenerator(model="Qwen/Qwen2.5-1.5B-Instruct", api_base_url=CHAT_URL))

rag.connect("text_embedder.embedding", "retriever.query_embedding")
rag.connect("retriever.documents", "ranker.documents")
rag.connect("ranker.documents", "prompt.documents")
rag.connect("prompt.prompt", "llm.messages")

🚅 Components
  - text_embedder: VLLMTextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - ranker: VLLMRanker
  - prompt: ChatPromptBuilder
  - llm: VLLMChatGenerator
🛤️ Connections
  - text_embedder.embedding -> retriever.query_embedding (list[float])
  - retriever.documents -> ranker.documents (list[Document])
  - ranker.documents -> prompt.documents (list[Document])
  - prompt.prompt -> llm.messages (list[ChatMessage])

## 4. Ask a question

In [5]:
question = "Where is the Eiffel Tower and when was it built?"

result = rag.run({
    "text_embedder": {"text": question},
    "ranker": {"query": question},
    "prompt": {"query": question},
})
print(result["llm"]["replies"][0].text)

The Eiffel Tower is located in Paris, France and was built in 1889.


## Summary

We built a complete RAG pipeline — embedding, retrieval, reranking, and generation — with **every model running on Intel GPUs** via vLLM, orchestrated by Haystack.

The key takeaway: **the Haystack pipeline code is identical regardless of the underlying hardware.** vLLM's OpenAI-compatible API means the same notebook runs on Intel Arc, Intel Data Center GPUs, or NVIDIA GPUs — only the `docker run` / `vllm serve` launch commands differ.

**Learn more:**
- [vLLM XPU installation](https://docs.vllm.ai/en/latest/getting_started/installation/gpu.html?device=xpu)
- [Intel vLLM container](https://hub.docker.com/r/intel/vllm)
- [`VLLMChatGenerator`](https://docs.haystack.deepset.ai/docs/vllmchatgenerator), [`VLLMTextEmbedder`](https://docs.haystack.deepset.ai/docs/vllmtextembedder), [`VLLMRanker`](https://docs.haystack.deepset.ai/docs/vllmranker)